# Pronóstico con Prophet: tendencia diaria y periodicidad horaria
Este notebook usa `Prophet` para estimar la tendencia del precio del oro (`GC=F`) y generar pronósticos de los próximos 7 días en dos escalas:
- `Diaria` (7 días)
- `Horaria` con periodicidad de minutos/hora y rango deslizante

In [31]:
import warnings
import pandas as pd
import numpy as np
import yfinance as yf
from prophet import Prophet
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [20]:
ticker = 'GC=F'
daily_data = yf.download(ticker, period='6mo', interval='1d', progress=False)
if isinstance(daily_data.columns, pd.MultiIndex):
    daily_data.columns = daily_data.columns.droplevel('Ticker')

daily_data = daily_data[['Close']].dropna().reset_index()
daily_data = daily_data.rename(columns={'Date': 'ds', 'Close': 'y'})
daily_data['ds'] = pd.to_datetime(daily_data['ds'])
if daily_data['ds'].dt.tz is not None:
    daily_data['ds'] = daily_data['ds'].dt.tz_convert(None)
print('Datos diarios cargados:', daily_data.shape)
daily_data.head()

Datos diarios cargados: (125, 2)


Price,ds,y
0,2025-11-17,4068.300049
1,2025-11-18,4061.300049
2,2025-11-19,4077.699951
3,2025-11-20,4056.500000
4,2025-11-21,4076.699951


In [21]:
hourly_data = yf.download(ticker, period='6mo', interval='60m', progress=False)
if hourly_data.empty:
    hourly_data = yf.download(ticker, period='60d', interval='60m', progress=False)
if isinstance(hourly_data.columns, pd.MultiIndex):
    hourly_data.columns = hourly_data.columns.droplevel('Ticker')

hourly_data = hourly_data[['Close']].dropna().reset_index()
hourly_data = hourly_data.rename(columns={'Datetime': 'ds', 'Date': 'ds', 'Close': 'y'})
hourly_data['ds'] = pd.to_datetime(hourly_data['ds'])
if hourly_data['ds'].dt.tz is not None:
    hourly_data['ds'] = hourly_data['ds'].dt.tz_convert(None)
print('Datos horarios cargados:', hourly_data.shape)
hourly_data.head()

Datos horarios cargados: (2780, 2)


Price,ds,y
0,2025-11-18 04:00:00,4011.000000
1,2025-11-18 05:00:00,4018.699951
2,2025-11-18 06:00:00,4008.300049
3,2025-11-18 07:00:00,4013.399902
4,2025-11-18 08:00:00,4036.800049


In [22]:
# Modelo Prophet para datos diarios
model_daily = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False)
model_daily.add_seasonality(name='monthly', period=30.5, fourier_order=6)
model_daily.fit(daily_data)
future_daily = model_daily.make_future_dataframe(periods=7, freq='D')
forecast_daily = model_daily.predict(future_daily)
forecast_daily[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

21:06:39 - cmdstanpy - INFO - Chain [1] start processing
21:06:39 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
127,2026-05-20,4537.181477,4350.966562,4707.846331
128,2026-05-21,4536.534248,4360.912380,4714.816481
129,2026-05-22,4558.423398,4381.611755,4734.199831
130,2026-05-23,4575.517578,4408.821764,4747.224185
131,2026-05-24,4450.132518,4274.047594,4622.014188


In [23]:
# Modelo Prophet para datos horarios (60 minutos)
model_hourly = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False)
model_hourly.fit(hourly_data)
future_hourly = model_hourly.make_future_dataframe(periods=7 * 24, freq='60min')
forecast_hourly = model_hourly.predict(future_hourly)
forecast_hourly[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

21:06:43 - cmdstanpy - INFO - Chain [1] start processing
21:06:44 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
2943,2026-05-24 22:00:00,4536.322616,4386.351532,4687.751256
2944,2026-05-24 23:00:00,4542.140978,4381.362429,4689.138148
2945,2026-05-25 00:00:00,4546.674232,4377.605952,4680.305095
2946,2026-05-25 01:00:00,4548.821704,4387.366525,4700.815512
2947,2026-05-25 02:00:00,4548.854705,4383.933154,4695.355143


In [30]:
# Gráfico interactivo diario con tendencia y banda de confianza
fig_daily = go.Figure()
fig_daily.add_trace(go.Scatter(x=daily_data['ds'], y=daily_data['y'], mode='lines', name='Actual', line=dict(color='blue')))
fig_daily.add_trace(go.Scatter(x=forecast_daily['ds'], y=forecast_daily['yhat'], mode='lines', name='Pronóstico', line=dict(color='red')))
fig_daily.add_trace(go.Scatter(x=forecast_daily['ds'], y=forecast_daily['yhat_upper'], mode='lines', name='IC superior', line=dict(color='red', width=1), opacity=0.3))
fig_daily.add_trace(go.Scatter(x=forecast_daily['ds'], y=forecast_daily['yhat_lower'], mode='lines', name='IC inferior', line=dict(color='red', width=1), fill='tonexty', fillcolor='rgba(255, 0, 0, 0.2)', opacity=0.3))
fig_daily.update_layout(title='Prophet: Pronóstico diario de los próximos 7 días', xaxis_title='Fecha', yaxis_title='Precio de cierre', xaxis_rangeslider_visible=True)
fig_daily.update_xaxes(tickformat='%Y-%m-%d')
fig_daily.show(renderer='vscode')

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [28]:
# Gráfico interactivo horario con periodicidad de minutos/hora
fig_hourly = go.Figure()
fig_hourly.add_trace(go.Scatter(x=hourly_data['ds'], y=hourly_data['y'], mode='lines', name='Actual', line=dict(color='blue', width=1)))
fig_hourly.add_trace(go.Scatter(x=forecast_hourly['ds'], y=forecast_hourly['yhat'], mode='lines', name='Pronóstico horario', line=dict(color='orange', width=2)))
fig_hourly.add_trace(go.Scatter(x=forecast_hourly['ds'], y=forecast_hourly['yhat_upper'], mode='lines', name='IC superior', line=dict(color='orange', width=1), opacity=0.3))
fig_hourly.add_trace(go.Scatter(x=forecast_hourly['ds'], y=forecast_hourly['yhat_lower'], mode='lines', name='IC inferior', line=dict(color='orange', width=1), fill='tonexty', fillcolor='rgba(255, 165, 0, 0.2)', opacity=0.3))
fig_hourly.update_layout(title='Prophet: Pronóstico horario con periodicidad de minutos', xaxis_title='Fecha y hora', yaxis_title='Precio de cierre', xaxis_rangeslider_visible=True, hovermode='x unified')
fig_hourly.update_xaxes(tickformat='%Y-%m-%d %H:%M', tickangle=45)
fig_hourly.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## Interpretación
- El primer gráfico muestra la tendencia diaria y el pronóstico de Prophet para los próximos 7 días.
- El segundo gráfico muestra la serie horaria con periodicidad de 60 minutos, la proyección de Prophet y la banda de confianza.
- El rango deslizante horizontal permite desplazar el eje de tiempo y ver los minutos/hora del pronóstico.